In [0]:
USE CATALOG rearc;
USE SCHEMA bronze;

In [0]:
CREATE OR REPLACE TABLE period
AS
SELECT *
FROM read_files(
    '/Volumes/rearc/raw/landing/pr.period',
    format => 'csv',
    header => true,
    delimiter => '\t'
);

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE series
AS
SELECT 
    TRIM(`series_id        `) AS series_id,
    sector_code,
    class_code,
    measure_code,
    duration_code,
    seasonal,
    base_year,
    footnote_codes,
    begin_year,
    begin_period,
    end_year,
    end_period,
    _rescued_data
FROM read_files(
    '/Volumes/rearc/raw/landing/pr.series',
    format => 'csv',
    header => true,
    delimiter => '\t'
);

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE class_lookup
AS
SELECT *
FROM read_files(
    '/Volumes/rearc/raw/landing/pr.class',
    format => 'csv',
    header => true,
    delimiter => '\t'
);

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE measure
AS
SELECT *
FROM read_files(
    '/Volumes/rearc/raw/landing/pr.measure',
    format => 'csv',
    header => true,
    delimiter => '\t'
);

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE sector
AS
SELECT *
FROM read_files(
    '/Volumes/rearc/raw/landing/pr.sector',
    format => 'csv',
    header => true,
    delimiter => '\t'
);

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE duration
AS
SELECT *
FROM read_files(
    '/Volumes/rearc/raw/landing/pr.duration',
    format => 'csv',
    header => true,
    delimiter => '\t'
);

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE seasonal
AS
SELECT *
FROM read_files(
    '/Volumes/rearc/raw/landing/pr.seasonal',
    format => 'csv',
    header => true,
    delimiter => '\t'
);

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE footnote
AS
SELECT *
FROM read_files(
    '/Volumes/rearc/raw/landing/pr.footnote',
    format => 'csv',
    header => true,
    delimiter => '\t'
);

num_affected_rows,num_inserted_rows


In [0]:
%python
with open("/Volumes/rearc/raw/landing/pr.data.1.AllData") as f:
    print(repr(f.readline()))

'series_id        \tyear\tperiod\t       value\tfootnote_codes\n'


In [0]:
CREATE OR REPLACE TABLE productivity_data AS
SELECT
    TRIM(`series_id        `) AS series_id,
    year,
    period,
    TRIM(`       value`) AS value,
    footnote_codes
FROM read_files(
    '/Volumes/rearc/raw/landing/pr.data.1.AllData',
    format => 'csv',
    header => true,
    delimiter => '\t'
);

num_affected_rows,num_inserted_rows


In [0]:
%python
df = spark.read.json("/Volumes/rearc/raw/landing/population.json")

df.printSchema()

display(df)

root
 |-- annotations: struct (nullable = true)
 |    |-- dataset_link: string (nullable = true)
 |    |-- dataset_name: string (nullable = true)
 |    |-- source_description: string (nullable = true)
 |    |-- source_name: string (nullable = true)
 |    |-- subtopic: string (nullable = true)
 |    |-- table_id: string (nullable = true)
 |    |-- topic: string (nullable = true)
 |-- columns: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- data: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- Nation: string (nullable = true)
 |    |    |-- Nation ID: string (nullable = true)
 |    |    |-- Population: double (nullable = true)
 |    |    |-- Year: long (nullable = true)
 |-- page: struct (nullable = true)
 |    |-- limit: long (nullable = true)
 |    |-- offset: long (nullable = true)
 |    |-- total: long (nullable = true)



annotations,columns,data,page
"List(http://www.census.gov/programs-surveys/acs/, ACS 1-year Estimate, The American Community Survey (ACS) is conducted by the US Census and sent to a portion of the population every year., Census Bureau, Demographics, B01003, Diversity)","List(Nation ID, Nation, Year, Population)","List(List(United States, 01000US, 3.16128839E8, 2013), List(United States, 01000US, 3.18857056E8, 2014), List(United States, 01000US, 3.21418821E8, 2015), List(United States, 01000US, 3.23127515E8, 2016), List(United States, 01000US, 3.25719178E8, 2017), List(United States, 01000US, 3.27167439E8, 2018), List(United States, 01000US, 3.28239523E8, 2019), List(United States, 01000US, 3.31893745E8, 2021), List(United States, 01000US, 3.33287562E8, 2022), List(United States, 01000US, 3.34914896E8, 2023), List(United States, 01000US, 3.4011099E8, 2024))","List(0, 0, 11)"


In [0]:
%python
from pyspark.sql.functions import explode

population_df = (
    spark.read.json("/Volumes/rearc/raw/landing/population.json")
         .select(explode("data").alias("record"))
         .select("record.*")
)

display(population_df)

Nation,Nation ID,Population,Year
United States,01000US,3.16128839E8,2013
United States,01000US,3.18857056E8,2014
United States,01000US,3.21418821E8,2015
United States,01000US,3.23127515E8,2016
United States,01000US,3.25719178E8,2017
United States,01000US,3.27167439E8,2018
United States,01000US,3.28239523E8,2019
United States,01000US,3.31893745E8,2021
United States,01000US,3.33287562E8,2022
United States,01000US,3.34914896E8,2023


In [0]:
%python
population_df = (
    population_df
        .withColumnRenamed("Nation ID", "nation_id")
        .withColumnRenamed("Nation", "nation")
        .withColumnRenamed("Population", "population")
        .withColumnRenamed("Year", "year")
)
population_df.write \
    .mode("overwrite") \
    .saveAsTable("rearc.bronze.population")

In [0]:
SELECT 'period' AS table_name, COUNT(*) AS rows FROM period
UNION ALL
SELECT 'series', COUNT(*) FROM series
UNION ALL
SELECT 'class_lookup', COUNT(*) FROM class_lookup
UNION ALL
SELECT 'measure', COUNT(*) FROM measure
UNION ALL
SELECT 'sector', COUNT(*) FROM sector
UNION ALL
SELECT 'duration', COUNT(*) FROM duration
UNION ALL
SELECT 'seasonal', COUNT(*) FROM seasonal
UNION ALL
SELECT 'footnote', COUNT(*) FROM footnote
UNION ALL
SELECT 'productivity_data', COUNT(*) FROM productivity_data;

table_name,rows
period,5
series,282
class_lookup,2
measure,22
sector,6
duration,3
seasonal,2
footnote,1
productivity_data,77126


In [0]:
SELECT COUNT(*) FROM population;
SELECT * FROM population;

nation,nation_id,population,year
United States,01000US,3.16128839E8,2013
United States,01000US,3.18857056E8,2014
United States,01000US,3.21418821E8,2015
United States,01000US,3.23127515E8,2016
United States,01000US,3.25719178E8,2017
United States,01000US,3.27167439E8,2018
United States,01000US,3.28239523E8,2019
United States,01000US,3.31893745E8,2021
United States,01000US,3.33287562E8,2022
United States,01000US,3.34914896E8,2023


In [0]:
USE CATALOG rearc;
USE SCHEMA bronze;

DROP TABLE IF EXISTS period;
DROP TABLE IF EXISTS sector;
DROP TABLE IF EXISTS measure;
DROP TABLE IF EXISTS class_lookup;
DROP TABLE IF EXISTS duration;
DROP TABLE IF EXISTS seasonal;
DROP TABLE IF EXISTS footnote;
DROP TABLE IF EXISTS series;
DROP TABLE IF EXISTS productivity_data;
DROP TABLE IF EXISTS population;